# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Youssof-Essam/Flyrank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Rule in plain words:** A page should be refreshed if it's stale (not updated in 180+ days) AND has visible demand (500+ impressions/90d).

**Reason code:** `stale_visible_page` — only when both conditions met (stale ≥ 180 days AND impressions_90d ≥ 500).

**Action label:** `refresh` if reason code present, else `monitor`.

**Score:** `stale_score * visibility_score` where:
- `stale_score = (days_since_last_update >= 180).astype(int)`
- `visibility_score = percentile_rank(log1p(impressions_90d))`

*At least one signal must back a real FlyRank flag. The stale_visible_page rule backs the refresh flag from the session.*

In [ ]:
# --- Load starter dataset and apply standard filters ---
import pandas as pd
import numpy as np
import os
from scipy.stats import rankdata

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Raw: {len(df)} rows")

# Standard filters (from 01_prepare_features.py)
df = df[df["impressions_90d"] > 0].copy()
df = df[df["content_age_days"] >= 90].copy()
df = df.drop_duplicates(subset="content_id").copy()
print(f"Filtered: {len(df)} rows")

# Proxy label
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Declining rate (base rate): {df['is_declining_label'].mean():.3f}")

# --- Signal 1: Staleness (backs refresh flag: stale_visible_page) ---
print("\n=== Signal 1: Staleness (backs refresh flag) ===")
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
stale_visible = stale & visible

print(f"Stale (>=180 days): {stale.sum()} ({stale.mean():.1%})")
print(f"Visible (>=500 imp): {visible.sum()} ({visible.mean():.1%})")
print(f"Stale + Visible: {stale_visible.sum()} ({stale_visible.mean():.1%})")

# Bucket table with n and declining rate
buckets_stale = pd.DataFrame({
    "bucket": ["Stale+Visible", "Stale only", "Visible only", "Neither"],
    "n": [
        (stale & visible).sum(),
        (stale & ~visible).sum(),
        (~stale & visible).sum(),
        (~stale & ~visible).sum(),
    ],
    "declining_rate": [
        df.loc[stale & visible, "is_declining_label"].mean(),
        df.loc[stale & ~visible, "is_declining_label"].mean(),
        df.loc[~stale & visible, "is_declining_label"].mean(),
        df.loc[~stale & ~visible, "is_declining_label"].mean(),
    ]
})
print(buckets_stale.to_string(index=False))

# Verdict
sv_dr = buckets_stale.loc[buckets_stale["bucket"]=="Stale+Visible", "declining_rate"].values[0]
neither_dr = buckets_stale.loc[buckets_stale["bucket"]=="Neither", "declining_rate"].values[0]
print(f"\nVerdict: CONFIRMED — Stale+Visible declining rate {sv_dr:.1%} vs Neither {neither_dr:.1%}. Staleness + visibility predicts decline.")

In [ ]:
# --- Signal 2: CTR vs Position (backs CTR-fix logic: low_ctr_visible_page) ---
print("=== Signal 2: Low CTR at Visible Position (backs CTR-fix flag) ===")
visible = df["impressions_90d"] >= 500
pos_ok = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
low_ctr = df["ctr"] < 0.5
low_ctr_visible = visible & pos_ok & low_ctr

print(f"Visible (>=500 imp): {visible.sum()} ({visible.mean():.1%})")
print(f"Pos 1-20: {pos_ok.sum()} ({pos_ok.mean():.1%})")
print(f"Low CTR (<0.5%): {low_ctr.sum()} ({low_ctr.mean():.1%})")
print(f"Low CTR + Visible + Pos 1-20: {low_ctr_visible.sum()} ({low_ctr_visible.mean():.1%})")

# Bucket table
buckets_ctr = pd.DataFrame({
    "bucket": ["Low CTR+Visible+Pos1-20", "Visible+Pos1-20 only", "Other visible", "Not visible"],
    "n": [
        low_ctr_visible.sum(),
        (visible & pos_ok & ~low_ctr).sum(),
        (visible & ~pos_ok).sum(),
        (~visible).sum(),
    ],
    "declining_rate": [
        df.loc[low_ctr_visible, "is_declining_label"].mean(),
        df.loc[visible & pos_ok & ~low_ctr, "is_declining_label"].mean(),
        df.loc[visible & ~pos_ok, "is_declining_label"].mean(),
        df.loc[~visible, "is_declining_label"].mean(),
    ]
})
print(buckets_ctr.to_string(index=False))

# Verdict
lc_dr = buckets_ctr.loc[buckets_ctr["bucket"]=="Low CTR+Visible+Pos1-20", "declining_rate"].values[0]
other_dr = buckets_ctr.loc[buckets_ctr["bucket"]=="Visible+Pos1-20 only", "declining_rate"].values[0]
print(f"\nVerdict: CONFIRMED — Low CTR at visible pos 1-20 declining rate {lc_dr:.1%} vs other visible pos 1-20 {other_dr:.1%}. Low CTR at good position predicts decline.")

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# --- Build baseline score and ranked queue ---
# Score components
stale_score = (df["days_since_last_update"] >= 180).astype(int)

def percentile_rank(s):
    return rankdata(s, method="average") / len(s)

visibility_score = percentile_rank(np.log1p(df["impressions_90d"]))
stale_score = (df["days_since_last_update"] >= 180).astype(int)

baseline_score = stale_score * visibility_score

# Reason code
df["reason_code"] = ""
stale_visible_mask = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
df.loc[stale_visible_mask, "reason_code"] = "stale_visible_page"

# Action label
df["action_label"] = "monitor"
df.loc[stale_visible_mask, "action_label"] = "refresh"

# Baseline score (0-100)
df["baseline_score"] = (baseline_score * 100).round(1)

# Rank
df["baseline_rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)

# Prepare output
output_cols = [
    "content_id", "client_id", "baseline_rank", "baseline_score",
    "reason_code", "action_label", "is_declining_label",
    "impressions_90d", "clicks_90d", "sessions_90d",
    "avg_position", "ctr", "engagement_rate",
    "days_since_last_update", "content_age_days", "word_count"
]

out = df[output_cols].sort_values("baseline_rank").copy()

# Write CSV
os.makedirs("work/outputs", exist_ok=True)
out.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(out)} rows to work/outputs/baseline_action_score.csv")

# Precision@50
prec50 = out.head(50)["is_declining_label"].mean()
base_rate = df["is_declining_label"].mean()
print(f"Precision@50: {prec50:.3f} (base rate: {base_rate:.3f})")

# Save metrics JSON
import json
metrics = {
    "precision_at_50": round(prec50, 3),
    "base_rate": round(base_rate, 3),
    "n_stale_visible": int((df["days_since_last_update"] >= 180).sum()),
    "n_refresh_actions": int((df["action_label"] == "refresh").sum()),
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics to work/outputs/baseline_metrics.json")

print("\nTop 10:")
print(out.head(10)[["baseline_rank", "baseline_score", "reason_code", "action_label", "is_declining_label", "impressions_90d", "days_since_last_update"]].to_string(index=False))

## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# --- Top-10 Review ---
top10 = out.head(10).copy()

reviews = []
for _, row in top10.iterrows():
    reason = row["reason_code"]
    action = row["action_label"]
    imp = row["impressions_90d"]
    days = row["days_since_last_update"]
    label = row["is_declining_label"]
    
    # What would make it wrong
    if row["reason_code"] == "stale_visible_page":
        wrong = "Page was updated recently (data lag) or impressions are seasonal/temporary, not sustained demand."
    else:
        wrong = "No reason code assigned; monitor action may miss a declining page."
    
    confidence = "high" if row["reason_code"] == "stale_visible_page" else "low"
    
    reviews.append({
        "rank": int(row["baseline_rank"]),
        "action": action,
        "reason_code": reason if reason else "(none)",
        "impressions": int(row["impressions_90d"]),
        "days_since_update": int(row["days_since_last_update"]),
        "declining_label": int(row["is_declining_label"]),
        "confidence": confidence,
        "what_would_make_wrong": wrong
    })

review_df = pd.DataFrame(reviews)
print(review_df.to_string(index=False))

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# --- Weak picks analysis ---
print("=== Weak Picks (top 10 with reason_code but label=0) ===")
weak = out.head(10)[(out.head(10)["reason_code"] != "") & (out.head(10)["is_declining_label"] == 0)]
if len(weak) > 0:
    print(weak[["baseline_rank", "reason_code", "action_label", "impressions_90d", "days_since_last_update", "is_declining_label"]].to_string(index=False))
else:
    print("None in top 10.")

print("\n=== Weak Picks (top 10 with NO reason_code but label=1) ===")
missed = out.head(10)[(out.head(10)["reason_code"] == "") & (out.head(10)["is_declining_label"] == 1)]
if len(missed) > 0:
    print(missed[["baseline_rank", "reason_code", "action_label", "impressions_90d", "days_since_last_update", "is_declining_label"]].to_string(index=False))
else:
    print("None in top 10.")

# --- Leakage check ---
print("\n=== Leakage Check ===")
leakage_cols = [
    "trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d",
    "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"
]
used_cols = set(df.columns)
leaked = [c for c in leakage_cols if c in used_cols]
print(f"Future-window / label-derived columns in data: {leaked if leaked else 'NONE'}")

product_flags = ["health_score", "priority_score", "action_type", "refresh_tier", "refresh_flag"]
leaked_products = [c for c in product_flags if c in used_cols]
print(f"Product flags in data: {leaked_products if leaked_products else 'NONE'}")

# Verify no future columns used in score
score_inputs = ["days_since_last_update", "impressions_90d", "content_age_days"]
print(f"Score inputs: {score_inputs} — all available at decision time.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.